# KADMON Optuna — QuickBundles + Sinkhorn

Les bundles sont compressés en centroïdes QuickBundles pondérés par la taille de leurs clusters, puis comparés avec Sinkhorn débiaisé. Cette configuration sert de baseline tractographique pour évaluer les autres méthodes de compression. Le classement utilise `global_distance_mm`; pour chaque bundle, l'objectif minimise le rapport entre la distance intra-identité et la distance moyenne inter-identité.

## 1. Imports et configuration

In [1]:
from pathlib import Path
import os
import sys
import warnings
from time import perf_counter

import numpy as np
import optuna
import pandas as pd
import torch
from IPython.display import display
from joblib import Parallel, delayed

WORKING_DIR = Path.cwd().resolve()
KADMON_ROOT = next(
    (path for path in (WORKING_DIR, *WORKING_DIR.parents) if (path / 'kadmon').is_dir() and (path / 'notebooks').is_dir()),
    None,
)
if KADMON_ROOT is None:
    raise RuntimeError(f'Racine KADMON introuvable depuis {WORKING_DIR}.')
NOTEBOOK_DIR = KADMON_ROOT / 'notebooks' / 'optuna'
BUNDLES_DIR = KADMON_ROOT / 'notebooks' / 'bundles'
STUDY_PATH = NOTEBOOK_DIR / 'studies' / 'optuna_reid.sqlite3'
if str(KADMON_ROOT) not in sys.path:
    sys.path.insert(0, str(KADMON_ROOT))

from kadmon.compression import compress_quickbundles
from kadmon.io import BundleCollection
from kadmon.protocol import HCP_REID_PROTOCOL
from kadmon.reid import aggregate_reid_metrics, bundle_reid_metrics, set_trial_metrics
from kadmon.optimization import StudyRunPolicy, create_reid_study, run_until_complete
from kadmon.cpu import evaluate_pair_cpu as evaluate_sinkhorn_pair_cpu
from kadmon.gpu import (
    evaluate_pair_gpu as evaluate_sinkhorn_pair_gpu,
    release_gpu_memory,
)
from kadmon.selection import select_best_reid_trial

optuna.logging.set_verbosity(optuna.logging.WARNING)
EXPERIMENT_NAME = 'quickbundles_sinkhorn'
N_TRIALS = 80
GPU_DTYPE = torch.float32
GPU_MDF_BATCH_SIZE = 128
GPU_DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
GPU_BACKEND_VALIDATED = False
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info(GPU_DEVICE)
    print(f'[GPU] backend=PyTorch/POT, device={torch.cuda.get_device_name(GPU_DEVICE)}')
    print(f'[GPU] mémoire={free / 2**30:.2f}/{total / 2**30:.2f} Gio, dtype={GPU_DTYPE}')
else:
    warnings.warn('CUDA indisponible : fallback CPU explicite.', RuntimeWarning)


Info: some functions in tractosearch.resampling are faster when 'numba' is installed
[GPU] backend=PyTorch/POT, device=NVIDIA GeForce RTX 5070 Ti
[GPU] mémoire=15.28/15.51 Gio, dtype=torch.float32


## 2. Données

Pour chaque bundle, l'acquisition `103818` est comparée aux dix acquisitions `_re`. `103818_re` constitue la comparaison intra-identité et les neuf autres acquisitions les comparaisons inter-identité.

In [2]:
REFERENCE_SUBJECT = HCP_REID_PROTOCOL.reference_subject
INTRA_IDENTITY_SUBJECT = HCP_REID_PROTOCOL.intra_identity_subject
COMPARISON_SUBJECTS = HCP_REID_PROTOCOL.comparison_subjects
SUBJECTS = HCP_REID_PROTOCOL.subjects
N_POINTS = HCP_REID_PROTOCOL.n_points
SEED = 42
BUNDLES_TO_RUN = None  # tuple de noms exacts pour un essai court
N_JOBS_CPU = min(8, os.cpu_count() or 1)
MAX_REPRESENTATIVES = 5000  # Pruner avant les matrices MDF/Sinkhorn

bundles = BundleCollection(
    BUNDLES_DIR, SUBJECTS, n_points=N_POINTS, selected=BUNDLES_TO_RUN
)
SUBJECT_FILES = bundles.files
bundle_names = bundles.names
bundle_cache = bundles.cache
compression_cache = {}

load_bundle = bundles.load

def compress_one(subject, bundle_name, threshold):
    distribution = compress_quickbundles(
        np.asarray(load_bundle(subject, bundle_name), dtype=np.float64),
        threshold=threshold,
    )
    return (subject, bundle_name, threshold), distribution

def prepare_compressions(threshold, selected_bundles=None):
    selected = tuple(bundle_names if selected_bundles is None else selected_bundles)
    missing = [
        (subject, bundle_name, threshold)
        for bundle_name in selected
        for subject in SUBJECTS
        if (subject, bundle_name, threshold) not in compression_cache
    ]
    computed = Parallel(
        n_jobs=min(N_JOBS_CPU, len(missing)), backend='threading'
    )(
        delayed(compress_one)(subject, bundle_name, threshold)
        for subject, bundle_name, _ in missing
    ) if missing else []
    compression_cache.update(computed)
    return {
        (subject, bundle_name): compression_cache[(subject, bundle_name, threshold)]
        for bundle_name in selected
        for subject in SUBJECTS
    }

print(f'Données : {BUNDLES_DIR}')
print('Protocole : 1 comparaison intra-identité et 9 inter-identité par bundle')
print(f'Bundles communs : {len(bundle_names)}; workers CPU : {N_JOBS_CPU}')
display(pd.DataFrame({'bundle': bundle_names}))


Données : /home/colin/Tractographie/KADMON/notebooks/bundles
Protocole : 1 comparaison intra-identité et 9 inter-identité par bundle
Bundles communs : 31; workers CPU : 8


,bundle
0,tractosearch_nn_8_0mm_all_AF_L_m
1,tractosearch_nn_8_0mm_all_AF_R_m
2,tractosearch_nn_8_0mm_all_CC_1_m
3,tractosearch_nn_8_0mm_all_CC_2a_m
4,tractosearch_nn_8_0mm_all_CC_2b_m
5,tractosearch_nn_8_0mm_all_CC_3_m
6,tractosearch_nn_8_0mm_all_CC_4_m
7,tractosearch_nn_8_0mm_all_CC_5_m
8,tractosearch_nn_8_0mm_all_CC_6_m
9,tractosearch_nn_8_0mm_all_CC_7_m


## 3. Espace de recherche

- `threshold`: 6 à 30 mm, par pas de 1 mm;
- `epsilon`: 0,001 à 1,0 sur une échelle logarithmique.

Le seuil QuickBundles contrôle directement le nombre de centroïdes : un seuil plus élevé produit une compression plus forte. `epsilon` contrôle la régularisation entropique et la diffusion du plan Sinkhorn.

In [3]:
def sample_parameters(trial):
    return (
        {'threshold': trial.suggest_float('threshold', 6.0, 30.0, step=1.0)},
        {
            'epsilon': trial.suggest_float('epsilon', 0.001, 1.0, log=True),
            'max_iter': 2000,
            'stop_threshold': 1e-6,
            'reject_threshold': 1e-5,
        },
    )


## 4. Objectif Optuna

In [4]:
def evaluate_pair_gpu(source_distribution, target_distribution, parameters, *, gpu_cache=None, return_cost=False, log=False):
    return evaluate_sinkhorn_pair_gpu(
        source_distribution,
        target_distribution,
        parameters,
        device=GPU_DEVICE,
        dtype=GPU_DTYPE,
        batch_size=GPU_MDF_BATCH_SIZE,
        self_cost_cache=gpu_cache,
        return_cost=return_cost,
        log=log,
    )

def evaluate_pair_cpu(source_distribution, target_distribution, parameters):
    return evaluate_sinkhorn_pair_cpu(source_distribution, target_distribution, parameters)

def validate_gpu_backend():
    global GPU_BACKEND_VALIDATED
    if not torch.cuda.is_available():
        return False
    pilot = min(bundle_names, key=lambda name: sum(len(load_bundle(s, name)) for s in SUBJECTS))
    distributions = prepare_compressions(11.0, [pilot])
    parameters = {
        'epsilon': 0.75, 'max_iter': 2000,
        'stop_threshold': 1e-6, 'reject_threshold': 1e-5,
    }
    rows, cpu_distances, gpu_distances = [], [], []
    for index, candidate in enumerate(COMPARISON_SUBJECTS):
        source_distribution = distributions[(REFERENCE_SUBJECT, pilot)]
        target_distribution = distributions[(candidate, pilot)]
        cpu = evaluate_pair_cpu(source_distribution, target_distribution, parameters)
        gpu = evaluate_pair_gpu(
            source_distribution, target_distribution, parameters,
            return_cost=True, log=index == 0,
        )
        difference = np.abs(cpu['cost'] - gpu['cost'])
        cpu_distances.append(cpu['global_distance_mm'])
        gpu_distances.append(gpu['global_distance_mm'])
        rows.append({
            'bundle': pilot,
            'candidate': candidate,
            'cost_max_abs': float(difference.max()),
            'cost_mean_abs': float(difference.mean()),
            'distance_abs': abs(cpu['global_distance_mm'] - gpu['global_distance_mm']),
            'objective_abs': abs(cpu['objective'] - gpu['objective']),
        })
    validation = pd.DataFrame(rows)
    display(validation)
    intra = COMPARISON_SUBJECTS.index(INTRA_IDENTITY_SUBJECT)
    cpu_ratio = cpu_distances[intra] / np.delete(cpu_distances, intra).mean()
    gpu_ratio = gpu_distances[intra] / np.delete(gpu_distances, intra).mean()
    GPU_BACKEND_VALIDATED = bool(
        validation['cost_max_abs'].max() <= 2e-4
        and np.allclose(cpu_distances, gpu_distances, rtol=5e-4, atol=1e-5)
        and np.isclose(cpu_ratio, gpu_ratio, rtol=5e-4, atol=1e-5)
    )
    if GPU_BACKEND_VALIDATED:
        print('[GPU] Validation CPU/GPU réussie; backend CUDA activé.')
    else:
        warnings.warn('Validation CPU/GPU échouée : fallback CPU.', RuntimeWarning)
    release_gpu_memory()
    return GPU_BACKEND_VALIDATED

def evaluate_bundle(bundle_name, distributions, parameters, use_gpu):
    pair_metrics = []
    gpu_cache = {}
    for index, candidate in enumerate(COMPARISON_SUBJECTS):
        source_distribution = distributions[(REFERENCE_SUBJECT, bundle_name)]
        target_distribution = distributions[(candidate, bundle_name)]
        if use_gpu:
            value = evaluate_pair_gpu(
                source_distribution, target_distribution, parameters,
                gpu_cache=gpu_cache, log=index == 0,
            )
        else:
            value = evaluate_pair_cpu(source_distribution, target_distribution, parameters)
        pair_metrics.append(value)
    gpu_cache.clear()
    return bundle_reid_metrics(
        bundle_name, [row['global_distance_mm'] for row in pair_metrics],
        comparison_subjects=COMPARISON_SUBJECTS,
        intra_identity_subject=INTRA_IDENTITY_SUBJECT,
        mean_displacement_mm=np.mean([row['mean_mm'] for row in pair_metrics]),
        mean_transported_mass=np.mean([row['mass'] for row in pair_metrics]),
        mean_n_representatives=np.mean([
            len(distributions[(subject, bundle_name)][0]) for subject in SUBJECTS
        ]),
    )

def objective(trial):
    started = perf_counter()
    compression_parameters, transport_parameters = sample_parameters(trial)
    distributions = prepare_compressions(compression_parameters['threshold'])
    representative_counts = {
        key: len(distribution[0])
        for key, distribution in distributions.items()
    }
    worst_key = max(representative_counts, key=representative_counts.get)
    worst_count = representative_counts[worst_key]
    trial.set_user_attr('max_n_representatives', int(worst_count))
    if worst_count > MAX_REPRESENTATIVES:
        compression_cache.clear()
        raise optuna.TrialPruned(
            f'Compression insuffisante pour {worst_key}: {worst_count} représentants '
            f'(limite {MAX_REPRESENTATIVES}).'
        )
    use_gpu = GPU_BACKEND_VALIDATED
    if not use_gpu:
        warnings.warn('[CPU fallback] CUDA absente ou validation échouée.', RuntimeWarning)
    try:
        results = [
            evaluate_bundle(name, distributions, transport_parameters, use_gpu)
            for name in bundle_names
        ]
    except RuntimeError as exc:
        if 'Sinkhorn' in str(exc) and ('convergé' in str(exc) or 'converg' in str(exc)):
            raise optuna.TrialPruned(str(exc)) from exc
        raise
    finally:
        compression_cache.clear()
        release_gpu_memory()
    bundle_metrics = pd.DataFrame(results)
    aggregates = aggregate_reid_metrics(
        bundle_metrics, n_comparison_subjects=len(COMPARISON_SUBJECTS),
        elapsed_s=perf_counter() - started,
        extra={'backend': 'PyTorch/POT CUDA' if use_gpu else 'CPU fallback'},
    )
    set_trial_metrics(trial, aggregates)
    return aggregates['mean_intra_inter_ratio']


## 5. Optimisation

La base SQLite partagée est l'unique sortie automatique de l'étude.

In [5]:
validate_gpu_backend()
study = create_reid_study(STUDY_PATH, EXPERIMENT_NAME, seed=SEED)
run_until_complete(
    study, objective, StudyRunPolicy(N_TRIALS, gc_after_trial=False)
)
print(f'Base SQLite : {STUDY_PATH}')


[GPU] Cost matrix: shape=(12, 10), dtype=torch.float32
[GPU] OT backend: PyTorch/POT


,bundle,candidate,cost_max_abs,cost_mean_abs,distance_abs,objective_abs
0,tractosearch_nn_8_0mm_all_IFOF_R_m,103818_re,0.000004,1.029174e-06,1.322936e-07,1.878876e-08
1,tractosearch_nn_8_0mm_all_IFOF_R_m,135528_re,0.000004,8.855547e-07,3.458354e-07,5.869850e-09
2,tractosearch_nn_8_0mm_all_IFOF_R_m,143325_re,0.000008,1.044500e-06,6.787094e-07,2.640167e-08
3,tractosearch_nn_8_0mm_all_IFOF_R_m,177746_re,0.000004,1.053015e-06,1.298629e-06,7.448145e-09
4,tractosearch_nn_8_0mm_all_IFOF_R_m,194140_re,0.000004,9.347995e-07,7.540602e-07,9.984023e-09
5,tractosearch_nn_8_0mm_all_IFOF_R_m,250427_re,0.000008,1.027471e-06,4.355170e-07,7.669176e-09
6,tractosearch_nn_8_0mm_all_IFOF_R_m,433839_re,0.000004,9.970231e-07,2.720004e-07,1.768607e-08
7,tractosearch_nn_8_0mm_all_IFOF_R_m,627549_re,0.000008,1.105997e-06,1.844140e-06,1.553520e-08
8,tractosearch_nn_8_0mm_all_IFOF_R_m,783462_re,0.000004,1.082818e-06,3.700466e-07,3.588345e-09
9,tractosearch_nn_8_0mm_all_IFOF_R_m,861456_re,0.000004,1.077299e-06,1.191890e-07,3.032397e-08


[GPU] Validation CPU/GPU réussie; backend CUDA activé.
Étude : 0/80 essais COMPLETE; cible restante=80.


  0%|          | 0/80 [00:00<?, ?it/s]

[GPU] Cost matrix: shape=(26, 25), dtype=torch.float32
[GPU] OT backend: PyTorch/POT
[GPU] Cost matrix: shape=(30, 28), dtype=torch.float32
[GPU] OT backend: PyTorch/POT
[GPU] Cost matrix: shape=(7, 5), dtype=torch.float32
[GPU] OT backend: PyTorch/POT
[GPU] Cost matrix: shape=(13, 14), dtype=torch.float32
[GPU] OT backend: PyTorch/POT
[GPU] Cost matrix: shape=(10, 12), dtype=torch.float32
[GPU] OT backend: PyTorch/POT
[GPU] Cost matrix: shape=(18, 17), dtype=torch.float32
[GPU] OT backend: PyTorch/POT
[GPU] Cost matrix: shape=(15, 13), dtype=torch.float32
[GPU] OT backend: PyTorch/POT
[GPU] Cost matrix: shape=(25, 25), dtype=torch.float32
[GPU] OT backend: PyTorch/POT
[GPU] Cost matrix: shape=(44, 41), dtype=torch.float32
[GPU] OT backend: PyTorch/POT
[GPU] Cost matrix: shape=(45, 47), dtype=torch.float32
[GPU] OT backend: PyTorch/POT
[GPU] Cost matrix: shape=(18, 19), dtype=torch.float32
[GPU] OT backend: PyTorch/POT
[GPU] Cost matrix: shape=(17, 17), dtype=torch.float32
[GPU] OT bac

...

[GPU] Cost matrix: shape=(5, 5), dtype=torch.float32
[GPU] OT backend: PyTorch/POT
[GPU] Cost matrix: shape=(4, 4), dtype=torch.float32
[GPU] OT backend: PyTorch/POT
[GPU] Cost matrix: shape=(1, 1), dtype=torch.float32
[GPU] OT backend: PyTorch/POT
[GPU] Cost matrix: shape=(3, 3), dtype=torch.float32
[GPU] OT backend: PyTorch/POT
[GPU] Cost matrix: shape=(1, 1), dtype=torch.float32
[GPU] OT backend: PyTorch/POT
[GPU] Cost matrix: shape=(1, 1), dtype=torch.float32
[GPU] OT backend: PyTorch/POT
[GPU] Cost matrix: shape=(2, 3), dtype=torch.float32
[GPU] OT backend: PyTorch/POT
[GPU] Cost matrix: shape=(3, 4), dtype=torch.float32
[GPU] OT backend: PyTorch/POT
[GPU] Cost matrix: shape=(7, 8), dtype=torch.float32
[GPU] OT backend: PyTorch/POT
[GPU] Cost matrix: shape=(10, 11), dtype=torch.float32
[GPU] OT backend: PyTorch/POT
[GPU] Cost matrix: shape=(4, 4), dtype=torch.float32
[GPU] OT backend: PyTorch/POT
[GPU] Cost matrix: shape=(3, 3), dtype=torch.float32
[GPU] OT backend: PyTorch/POT
[G

  0%|          | 0/4 [00:00<?, ?it/s]

[GPU] Cost matrix: shape=(3, 3), dtype=torch.float32
[GPU] OT backend: PyTorch/POT
[GPU] Cost matrix: shape=(3, 3), dtype=torch.float32
[GPU] OT backend: PyTorch/POT
[GPU] Cost matrix: shape=(1, 1), dtype=torch.float32
[GPU] OT backend: PyTorch/POT
[GPU] Cost matrix: shape=(2, 2), dtype=torch.float32
[GPU] OT backend: PyTorch/POT
[GPU] Cost matrix: shape=(1, 1), dtype=torch.float32
[GPU] OT backend: PyTorch/POT
[GPU] Cost matrix: shape=(1, 1), dtype=torch.float32
[GPU] OT backend: PyTorch/POT
[GPU] Cost matrix: shape=(1, 1), dtype=torch.float32
[GPU] OT backend: PyTorch/POT
[GPU] Cost matrix: shape=(1, 1), dtype=torch.float32
[GPU] OT backend: PyTorch/POT
[GPU] Cost matrix: shape=(5, 5), dtype=torch.float32
[GPU] OT backend: PyTorch/POT
[GPU] Cost matrix: shape=(6, 6), dtype=torch.float32
[GPU] OT backend: PyTorch/POT
[GPU] Cost matrix: shape=(2, 2), dtype=torch.float32
[GPU] OT backend: PyTorch/POT
[GPU] Cost matrix: shape=(2, 3), dtype=torch.float32
[GPU] OT backend: PyTorch/POT
[GPU

  0%|          | 0/2 [00:00<?, ?it/s]

[GPU] Cost matrix: shape=(3, 3), dtype=torch.float32
[GPU] OT backend: PyTorch/POT
[GPU] Cost matrix: shape=(4, 4), dtype=torch.float32
[GPU] OT backend: PyTorch/POT
[GPU] Cost matrix: shape=(1, 1), dtype=torch.float32
[GPU] OT backend: PyTorch/POT
[GPU] Cost matrix: shape=(2, 2), dtype=torch.float32
[GPU] OT backend: PyTorch/POT
[GPU] Cost matrix: shape=(1, 1), dtype=torch.float32
[GPU] OT backend: PyTorch/POT
[GPU] Cost matrix: shape=(1, 1), dtype=torch.float32
[GPU] OT backend: PyTorch/POT
[GPU] Cost matrix: shape=(1, 1), dtype=torch.float32
[GPU] OT backend: PyTorch/POT
[GPU] Cost matrix: shape=(2, 1), dtype=torch.float32
[GPU] OT backend: PyTorch/POT
[GPU] Cost matrix: shape=(6, 6), dtype=torch.float32
[GPU] OT backend: PyTorch/POT
[GPU] Cost matrix: shape=(7, 6), dtype=torch.float32
[GPU] OT backend: PyTorch/POT
[GPU] Cost matrix: shape=(2, 2), dtype=torch.float32
[GPU] OT backend: PyTorch/POT
[GPU] Cost matrix: shape=(2, 3), dtype=torch.float32
[GPU] OT backend: PyTorch/POT
[GPU

## 6. Analyse des essais observés

In [6]:
trials_df = study.trials_dataframe(attrs=('number', 'value', 'params', 'user_attrs', 'state'))
complete = trials_df[trials_df['state'] == 'COMPLETE'].dropna(subset=['value'])
display(complete.sort_values('value').head(10))

best = study.best_trial
display(pd.Series({'experiment': EXPERIMENT_NAME, 'trial': best.number,
                   'mean_intra_inter_ratio': best.value, **best.params,
                   **best.user_attrs}, name='meilleur essai').to_frame())
print('RE-ID validée :', best.user_attrs.get('reid_valid_bundles', []))
print('RE-ID échouée :', best.user_attrs.get('reid_failed_bundles', []))

reid_best = select_best_reid_trial(study.trials)
display(pd.Series({'experiment': EXPERIMENT_NAME, 'trial': reid_best.number,
                   'selection': 'Top-1, rang, ratio, couverture',
                   'mean_intra_inter_ratio': reid_best.value, **reid_best.params,
                   **reid_best.user_attrs}, name='meilleur essai RE-ID').to_frame())
print('RE-ID sélectionnée — validée :', reid_best.user_attrs.get('reid_valid_bundles', []))
print('RE-ID sélectionnée — échouée :', reid_best.user_attrs.get('reid_failed_bundles', []))

optuna.visualization.plot_param_importances(study).show()
optuna.visualization.plot_contour(study, params=['epsilon', 'threshold']).show()


,number,value,params_epsilon,params_threshold,user_attrs_backend,user_attrs_elapsed_s,user_attrs_intra_identity_top1_accuracy,user_attrs_max_n_representatives,user_attrs_mean_displacement_mm,user_attrs_mean_inter_identity_distance_mm,...,user_attrs_mean_intra_inter_ratio,user_attrs_mean_intra_inter_separation_margin_mm,user_attrs_mean_n_representatives,user_attrs_mean_transported_mass,user_attrs_median_intra_inter_ratio,user_attrs_n_bundles,user_attrs_n_comparisons,user_attrs_reid_failed_bundles,user_attrs_reid_valid_bundles,state
125,125,0.382318,0.125785,29.0,PyTorch/POT CUDA,54.264900,0.806452,9,5.226941,6.483119,...,0.382318,1.261792,2.307918,1.0,0.330483,31.0,310.0,"[tractosearch_nn_8_0mm_all_CC_6_m, tractosearc...","[tractosearch_nn_8_0mm_all_AF_L_m, tractosearc...",COMPLETE
79,79,0.382337,0.125978,29.0,PyTorch/POT CUDA,50.876212,0.806452,9,5.227064,6.483623,...,0.382337,1.261782,2.307918,1.0,0.330669,31.0,310.0,"[tractosearch_nn_8_0mm_all_CC_6_m, tractosearc...","[tractosearch_nn_8_0mm_all_AF_L_m, tractosearc...",COMPLETE
75,75,0.382617,0.128665,29.0,PyTorch/POT CUDA,49.500108,0.806452,9,5.228859,6.490745,...,0.382617,1.261618,2.307918,1.0,0.333317,31.0,310.0,"[tractosearch_nn_8_0mm_all_CC_6_m, tractosearc...","[tractosearch_nn_8_0mm_all_AF_L_m, tractosearc...",COMPLETE
98,98,0.382737,0.129767,29.0,PyTorch/POT CUDA,48.055950,0.806452,9,5.229633,6.493719,...,0.382737,1.261544,2.307918,1.0,0.334429,31.0,310.0,"[tractosearch_nn_8_0mm_all_CC_6_m, tractosearc...","[tractosearch_nn_8_0mm_all_AF_L_m, tractosearc...",COMPLETE
63,63,0.382764,0.130012,29.0,PyTorch/POT CUDA,48.239558,0.806452,9,5.229808,6.494386,...,0.382764,1.261527,2.307918,1.0,0.334680,31.0,310.0,"[tractosearch_nn_8_0mm_all_CC_6_m, tractosearc...","[tractosearch_nn_8_0mm_all_AF_L_m, tractosearc...",COMPLETE
81,81,0.382880,0.131047,29.0,PyTorch/POT CUDA,49.024018,0.806452,9,5.230561,6.497218,...,0.382880,1.261454,2.307918,1.0,0.335741,31.0,310.0,"[tractosearch_nn_8_0mm_all_CC_6_m, tractosearc...","[tractosearch_nn_8_0mm_all_AF_L_m, tractosearc...",COMPLETE
117,117,0.383146,0.133324,29.0,PyTorch/POT CUDA,46.463383,0.806452,9,5.232288,6.503551,...,0.383146,1.261283,2.307918,1.0,0.338123,31.0,310.0,"[tractosearch_nn_8_0mm_all_CC_6_m, tractosearc...","[tractosearch_nn_8_0mm_all_AF_L_m, tractosearc...",COMPLETE
64,64,0.383399,0.135387,29.0,PyTorch/POT CUDA,44.530846,0.806452,9,5.233936,6.509408,...,0.383399,1.261116,2.307918,1.0,0.340330,31.0,310.0,"[tractosearch_nn_8_0mm_all_CC_6_m, tractosearc...","[tractosearch_nn_8_0mm_all_AF_L_m, tractosearc...",COMPLETE
62,62,0.383941,0.139536,29.0,PyTorch/POT CUDA,40.198315,0.806452,9,5.237491,6.521528,...,0.383941,1.260748,2.307918,1.0,0.344900,31.0,310.0,"[tractosearch_nn_8_0mm_all_CC_6_m, tractosearc...","[tractosearch_nn_8_0mm_all_AF_L_m, tractosearc...",COMPLETE
84,84,0.387775,0.162703,29.0,PyTorch/POT CUDA,31.463140,0.806452,9,5.262983,6.597158,...,0.387775,1.258116,2.307918,1.0,0.371961,31.0,310.0,"[tractosearch_nn_8_0mm_all_CC_6_m, tractosearc...","[tractosearch_nn_8_0mm_all_AF_L_m, tractosearc...",COMPLETE


,meilleur essai
experiment,quickbundles_sinkhorn
trial,125
mean_intra_inter_ratio,0.382318
threshold,29.0
epsilon,0.125785
backend,PyTorch/POT CUDA
elapsed_s,54.2649
intra_identity_top1_accuracy,0.806452
max_n_representatives,9
mean_displacement_mm,5.226941


RE-ID validée : ['tractosearch_nn_8_0mm_all_AF_L_m', 'tractosearch_nn_8_0mm_all_AF_R_m', 'tractosearch_nn_8_0mm_all_CC_1_m', 'tractosearch_nn_8_0mm_all_CC_2a_m', 'tractosearch_nn_8_0mm_all_CC_2b_m', 'tractosearch_nn_8_0mm_all_CC_3_m', 'tractosearch_nn_8_0mm_all_CC_4_m', 'tractosearch_nn_8_0mm_all_CC_5_m', 'tractosearch_nn_8_0mm_all_CC_7_m', 'tractosearch_nn_8_0mm_all_CG_L_m', 'tractosearch_nn_8_0mm_all_CST_R_m', 'tractosearch_nn_8_0mm_all_ICP_R_m', 'tractosearch_nn_8_0mm_all_ILF_L_m', 'tractosearch_nn_8_0mm_all_ILF_R_m', 'tractosearch_nn_8_0mm_all_MCP_m', 'tractosearch_nn_8_0mm_all_OR_L_m', 'tractosearch_nn_8_0mm_all_OR_R_m', 'tractosearch_nn_8_0mm_all_SLF_1_L_m', 'tractosearch_nn_8_0mm_all_SLF_1_R_m', 'tractosearch_nn_8_0mm_all_SLF_2_L_m', 'tractosearch_nn_8_0mm_all_SLF_2_R_m', 'tractosearch_nn_8_0mm_all_SLF_3_L_m', 'tractosearch_nn_8_0mm_all_SLF_3_R_m', 'tractosearch_nn_8_0mm_all_UF_L_m', 'tractosearch_nn_8_0mm_all_UF_R_m']
RE-ID échouée : ['tractosearch_nn_8_0mm_all_CC_6_m', 'tracto

## 7. Interprétation des résultats

L'étude finale contient **80 essais `COMPLETE`** et **50 essais `PRUNED`**; seuls les essais complets sont classés.

Le **trial 125** est l'optimum strict du ratio intra/inter (`score=0,382318`, `threshold=29 mm`, `epsilon≈0,125785`). Il ne produit toutefois qu'environ **2,31 représentants** par bundle et son Top-1 est de **80,65 %** (25/31), avec un rang moyen de **1,48**. Comme pour QuickBundles + Partial OT, l'objectif continu favorise donc ici une compression globale extrême qui ne maximise pas la réussite Top-1.

Selon la règle RE-ID commune, le **trial 108** est retenu (`threshold=14 mm`, `epsilon≈0,131023`) : Top-1 **90,32 %** (28/31), rang moyen **1,16**, ratio **0,654113** et environ **18,72 représentants** par bundle. Les échecs sont `CST_L`, `IFOF_L` et `IFOF_R`.

Deux autres essais atteignent le même Top-1 : le trial 120 (`threshold=11 mm`, environ 46,37 représentants) a le même rang moyen mais un ratio plus élevé (`0,776848`), tandis que le trial 114 (`threshold=6 mm`, `epsilon≈0,087448`, environ 537,56 représentants) a un rang moyen légèrement moins bon (`1,19`) et un ratio de `0,698405`. Le trial 108 gagne selon le classement RE-ID, mais le **trial 114 est retenu comme défaut anatomique** grâce à sa résolution beaucoup plus fine et à sa masse Sinkhorn complète.

- Pour la **meilleure RE-ID observée**, utiliser le trial 108.
- Pour les **analyses anatomiques exécutées par défaut dans KADMON**, utiliser le trial 114.
- Le trial 125 reste la référence de l'**optimum du ratio continu**, mais pas un défaut anatomique pertinent.
- Le seuil contrôle la résolution géométrique et `epsilon` la diffusion du plan; le nombre réel de représentants doit accompagner toute interprétation anatomique locale.
- Les corrélations observées ne suffisent pas à attribuer causalement les performances au seul nombre de représentants, car `threshold` et `epsilon` varient conjointement.